In [1]:
import numpy as np
import pandas as pd
from scipy.stats import linregress
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import mne
import os
import sys 
sys.path.append(os.path.abspath('..'))
from notifmmn import erputils,datautils
from glob import glob
from os import path 
rawdir =r"E:\ecf-exp2-notif-mmn\data\Processed"
resultdir = '../data/results/'

triggers = dict(
    S14='BN STD',
    S15='SN DEV',
    S24='SN STD',
    S25='BN DEV',
    S34='CN STD',
    S35='BN DEV',
)

ch_order = ['TP10', 'CP5', 'C5', 'FC5', 'C6', 'T8', 'C3', 'TP8', 'C4', 'C1', 'P5',
       'FT8', 'CP3', 'F7', 'TP9', 'FC1', 'C2', 'F3', 'FC6', 'TP7', 'F2', 'Fz',
       'FC3', 'Cz', 'P7', 'F1', 'FC2', 'AF3', 'Fp1', 'AFz', 'FC4', 'FCz', 'P8',
       'Fp2', 'FT7', 'P1', 'F8', 'AF4', 'CP2', 'F4', 'P3', 'CPz', 'T7', 'CP1',
       'Pz', 'CP4', 'F5', 'P4', 'PO7', 'CP6', 'AF8', 'AF7', 'FT9', 'P2', 'PO8',
       'P6', 'PO4', 'PO3', 'FT10', 'POz', 'F6', 'O1', 'O2', 'Oz']

%matplotlib widget
participants= pd.read_excel(f'E:/ecf-exp2-notif-mmn/data/Processed/Participant_details.xlsx')
delays = participants['Tone_delay'].rename('tone delay')
# list all preprocessed files
# preprocessing involves manual curation, band pass filtering,
# down sampling, removal of eye artifacts, and epoching
files = glob(path.join(rawdir, 'preprocessed_mark_bad_epoch', '*'))
print(len(files))
sorted(files)

52


['E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd011-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd012-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd013-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd014-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd015-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd016-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd017-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd018-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd019-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd020-epo.fif',
 'E:\\ecf-exp2-notif-mmn\\data\\Processed\\preprocessed_mark_bad_epoch\\sub-sd02

In [5]:
delays

0     0.008617
1     0.036576
2     0.008617
3     0.034830
4     0.006780
5     0.036576
6     0.006485
7     0.008617
8     0.001927
9     0.031429
10    0.006689
11    0.006780
12    0.006780
13    0.011837
14    0.007211
15    0.006485
16    0.031429
17    0.001927
18    0.009365
19    0.008141
20    0.001927
21    0.034830
22    0.031429
23    0.197551
24    0.001927
25    0.036576
26    0.001293
27    0.036576
28    0.071837
29    0.071837
30    0.006780
31    0.008141
32    0.001927
33    0.034830
34    0.001927
35    0.006485
36    0.006485
37    0.008617
38    0.011837
39    0.197551
40    0.036576
41    0.001927
42    0.001293
43    0.006485
44    0.034830
45    0.008617
46    0.014921
47    0.000975
48    0.197551
49    0.002925
50    0.008617
51    0.001293
52    0.006485
53    0.008617
54    0.001927
55    0.010091
56    0.036576
Name: tone delay, dtype: float64

In [ ]:
participants['participant_id']

,participant_id,age,is_controlled_block,gender,profession,b1_deviant,b2_deviant,b3_deviant,comments,SN_tone,Tone_delay
0,sd011,30,N,M,PG_acd,SN,BT,NaN,FT10 noisy channel interpolated and further r...,Hello_inMoto,0.008617
1,sd012,25,N,M,PG_acd,SN,BT,NaN,No bad channel,Whoop_doop,0.036576
2,sd013,25,N,M,PG_acd,SN,BT,NaN,TP10 noisy channel interpolated and further r...,Hello_inMoto,0.008617
3,sd014,28,N,M,PG_acd,SN,BT,NaN,bad electrode = None Bad subject only very few...,droplet,0.034830
4,sd015,31,N,M,PG_acd,SN,BT,NaN,bad electrode = None,realme_fun,0.006780
5,sd016,28,N,M,PG_acd,SN,BT,NaN,NaN,Whoop_doop,0.036576
6,sd017,25,N,M,PG_acd,SN,BT,NaN,NaN,spaceline,0.006485
7,sd018,28,N,M,PG_acd,SN,BT,NaN,NaN,Hello_inMoto,0.008617
8,sd019,25,N,M,PG_acd,SN,BT,NaN,NaN,iphone_note,0.001927
9,sd020,26,N,M,PG_acd,SN,BT,NaN,NaN,oneplustune,0.031429


In [7]:
sta_scores = {}
from tqdm import tqdm
for fname in tqdm(files):
        sub = path.basename(fname).split('-')[1]
        print(sub)
   
        _epochs = mne.read_epochs(fname, verbose='ERROR').apply_baseline(verbose='ERROR')
        _epochs = datautils.epochs_to_df_with_annotations(_epochs)

        n1_win = np.array([0.16, 0.28]) # n1 will be searched in this window, adjusted for the tone onset time
        sta_scores[sub] = pd.concat({
            c : erputils.compute_single_erp_scores(
                [_epochs.xs(True, level='valid').xs(c, level='condition').groupby(['cindex']).mean().T],
                window=n1_win if c in [14, 25, 35] else n1_win+participants[participants['participant_id']==sub]['Tone_delay'], latency_fractions=[0.25], extrema=-1
            ) for c in [14, 15, 24, 25]
        })
        sta_scores[sub].loc[(14, 0)] = sta_scores[sub].loc[(25, 0)]
        sta_scores[sub].loc[(24, 0)] = sta_scores[sub].loc[(15, 0)]

sta_scores = pd.concat(sta_scores, names=['subject', 'stimulus', 'index'])

# sta_scores = sta_scores.groupby(['stimulus', 'index']).mean()

f, ax = plt.subplots(figsize=(6, 3), tight_layout=True)
ax2 = ax#.twinx()

sta_scores.loc[14]['PA'][:8].plot(ax=ax, marker='.', lw=0.5)
sta_scores.loc[24]['PA'][:8].plot(ax=ax2, marker='.', lw=0.5, c='C1')

ax.set_xlabel('epoch post deviant')
ax.set_ylabel('BN N1 amplitude ($\mu V$)', c='C0')
ax2.set_ylabel('SN N1 amplitude ($\mu V$)', c='C1')
ax.set_xticks(range(8))
ax.set_xticklabels(['DEV', 1, 2, 3, 4, 5, 6, 7]);

<>:31: SyntaxWarning: invalid escape sequence '\m'
<>:32: SyntaxWarning: invalid escape sequence '\m'
<>:31: SyntaxWarning: invalid escape sequence '\m'
<>:32: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Prakash\AppData\Local\Temp\ipykernel_26876\1119640232.py:31: SyntaxWarning: invalid escape sequence '\m'
  ax.set_ylabel('BN N1 amplitude ($\mu V$)', c='C0')
C:\Users\Prakash\AppData\Local\Temp\ipykernel_26876\1119640232.py:32: SyntaxWarning: invalid escape sequence '\m'
  ax2.set_ylabel('SN N1 amplitude ($\mu V$)', c='C1')
  0%|          | 0/52 [00:00<?, ?it/s]

sd011


  0%|          | 0/52 [00:03<?, ?it/s]
C:\Users\Prakash\AppData\Local\Temp\ipykernel_26876\1119640232.py:31: SyntaxWarning: invalid escape sequence '\m'
  ax.set_ylabel('BN N1 amplitude ($\mu V$)', c='C0')
C:\Users\Prakash\AppData\Local\Temp\ipykernel_26876\1119640232.py:32: SyntaxWarning: invalid escape sequence '\m'
  ax2.set_ylabel('SN N1 amplitude ($\mu V$)', c='C1')


ValueError: Length of values (2) does not match length of index (1)

In [ ]:
sta_scores

{}